# Phase 2: model code, baselines and pilot run

Run on Google Colab with Google Drive mounted. Each phase starts only if the previous phase's gate file says GO. See the repository README for the full workflow.

### Cell 2.0: Start the session

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json, shutil, importlib, time, math
from pathlib import Path
import numpy as np, pandas as pd, torch

PROJECT = Path("/content/drive/MyDrive/ecg_leak")
sys.path.insert(0, str(PROJECT / "src"))
import ecgleak.config as ecfg
importlib.reload(ecfg)
CFG = ecfg.load(str(PROJECT / "config.yaml"))
PATHS = CFG["paths"]

gate1 = json.load(open(f"{PATHS['gates']}/gate_1.json"))
assert gate1["go"], "Phase 1 gate is not GO: finish Phase 1 first"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Phase 1 gate: GO | Device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU: Runtime > Change runtime type > T4 GPU")

### Cell 2.1: Load the heartbeat cache (lockbox sealed)

In [ ]:
LOCAL_CACHE = "/content/cache"
os.makedirs(LOCAL_CACHE, exist_ok=True)
splits = pd.read_csv(f"{PATHS['splits']}/splits.csv", index_col="ecg_id")
D = {}
for s in ["id_train", "id_eval", "probe_dev"]:          # lockbox stays sealed until Phase 3
    src, dst = f"{PATHS['cache']}/{s}.npz", f"{LOCAL_CACHE}/{s}.npz"
    if not os.path.isfile(dst):
        shutil.copyfile(src, dst)
    z = np.load(dst, allow_pickle=False)
    D[s] = {k: z[k] for k in z.files}
    D[s]["date"] = pd.to_datetime(splits.loc[D[s]["ecg_id"], "recording_date"]).to_numpy()
    print(f"{s:10s} records {len(D[s]['ecg_id']):6,} | patients {len(np.unique(D[s]['patient_id'])):5,} | signal {D[s]['signal'].shape}")
SUPER = list(D["probe_dev"]["label_names"])

### Cell 2.2: Model code: models.py

In [ ]:
%%writefile /content/drive/MyDrive/ecg_leak/src/ecgleak/models.py
import math
import torch, torch.nn as nn, torch.nn.functional as F

class ECGEncoder(nn.Module):
    """1-lead ECG window (B x 1250) -> 256-d unit vector (the template)."""
    def __init__(self, channels=(32, 64, 128), kernel=7, layers=2, heads=4, emb_dim=256, max_len=1024):
        super().__init__()
        blocks, c_in = [], 1
        for c in channels:
            blocks += [nn.Conv1d(c_in, c, kernel, padding=kernel // 2), nn.BatchNorm1d(c), nn.GELU(), nn.MaxPool1d(2)]
            c_in = c
        self.stem = nn.Sequential(*blocks)
        d = channels[-1]
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4 * d, dropout=0.1,
                                           activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.proj = nn.Linear(d, emb_dim)
        self.emb_dim = emb_dim

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        h = self.stem(x).transpose(1, 2)
        h = self.transformer(h + self.pos[:, :h.size(1)])
        return F.normalize(self.proj(h.mean(dim=1)), dim=1)

class ArcFaceHead(nn.Module):
    """Additive angular margin: pushes each person's templates into a tight cone."""
    def __init__(self, emb_dim, n_classes, s=30.0, m=0.5):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_classes, emb_dim) * 0.01)
        self.s, self.m = s, m
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m

    def forward(self, emb, y):
        cos = F.linear(emb, F.normalize(self.W, dim=1)).clamp(-1 + 1e-7, 1 - 1e-7)
        t = cos.gather(1, y[:, None])
        with_margin = torch.cos(torch.acos(t) + self.m)
        t_new = torch.where(t > self.th, with_margin, t - self.mm)
        return self.s * cos.scatter(1, y[:, None], t_new)

def build_encoder(cfg):
    m = cfg["model"]
    return ECGEncoder(m["cnn_channels"], m["kernel"], m["transformer_layers"], m["heads"], m["emb_dim"])

def untrained_copies(cfg, seeds, device):
    """Same architecture, random weights, never trained: the baseline for Delta_rand."""
    out = []
    for s in seeds:
        torch.manual_seed(s)
        out.append(build_encoder(cfg).to(device).eval())
    return out

### Cell 2.3: Training code: train.py (Phase 2 version)

In [ ]:
%%writefile /content/drive/MyDrive/ecg_leak/src/ecgleak/train.py
import glob, math, os, time
import numpy as np, torch, torch.nn.functional as F
from .models import build_encoder, ArcFaceHead

def _schedule(opt, total, warm):
    def f(step):
        if step < warm:
            return (step + 1) / max(1, warm)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm))))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def latest_checkpoint(run_dir):
    files = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "epoch_*.pt")))
    return files[-1] if files else None

def train_identity(cfg, signals, patient_ids, run_dir, device, epochs=None, eval_fn=None, crops_per_record=4,
                   seed=None, stop_after=None, max_steps_per_epoch=None, log=print):
    """ArcFace identity training with an automatic resume from the newest checkpoint in run_dir."""
    t, seed = cfg["train"], cfg["seed"] if seed is None else seed
    epochs = epochs or t["epochs"]
    window = int(cfg["preprocess"]["window_s"] * cfg["data"]["fs"])
    classes, y_all = np.unique(patient_ids, return_inverse=True)
    n, L, B = len(signals), signals.shape[1], t["batch"]
    steps = math.ceil(n * crops_per_record / B)
    if max_steps_per_epoch:
        steps = min(steps, max_steps_per_epoch)

    torch.manual_seed(seed)
    model = build_encoder(cfg).to(device)
    head = ArcFaceHead(cfg["model"]["emb_dim"], len(classes), cfg["arcface"]["s"], cfg["arcface"]["m"]).to(device)
    opt = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=t["lr"], weight_decay=t["weight_decay"])
    sched = _schedule(opt, steps * epochs, int(t["warmup_frac"] * steps * epochs))
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)

    start, history, best, bad = 0, [], float("inf"), 0
    ck = latest_checkpoint(run_dir)
    if ck:
        st = torch.load(ck, map_location=device, weights_only=False)
        model.load_state_dict(st["model"]); head.load_state_dict(st["head"])
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        torch.set_rng_state(st["torch_rng"].cpu())
        if device == "cuda" and st.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all([s.cpu() for s in st["cuda_rng"]])
        start, history, best, bad = st["epoch"] + 1, st["history"], st["best"], st["bad"]
        log(f"Resumed from {os.path.basename(ck)} -> continuing at epoch {start + 1}")

    for epoch in range(start, epochs):
        t0 = time.time()
        model.train(); head.train()
        rng = np.random.default_rng(seed * 1000 + epoch)
        idx = np.repeat(np.arange(n), crops_per_record); rng.shuffle(idx)
        offs = rng.integers(0, L - window + 1, size=len(idx))
        scale = rng.uniform(0.9, 1.1, size=len(idx))
        tot_loss = tot_acc = 0.0
        for b in range(steps):
            sl = slice(b * B, (b + 1) * B)
            ii = idx[sl]
            xb = np.stack([signals[i, o:o + window] for i, o in zip(ii, offs[sl])]).astype(np.float32)
            xb = (xb * scale[sl, None] + rng.normal(0, 0.05, xb.shape)).astype(np.float32)
            xb = torch.from_numpy(xb).to(device)
            yb = torch.from_numpy(y_all[ii]).long().to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                emb = model(xb)
            logits = head(emb.float(), yb)
            loss = F.cross_entropy(logits, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(head.parameters()), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot_loss += loss.item()
            tot_acc += (logits.argmax(1) == yb).float().mean().item()
        rec = {"epoch": epoch + 1, "loss": tot_loss / steps, "train_acc": tot_acc / steps,
               "lr": sched.get_last_lr()[0], "seconds": round(time.time() - t0, 1)}
        if eval_fn is not None:
            rec.update(eval_fn(model))
        improved = rec.get("eer", -epoch) < best
        if improved:
            best, bad = rec.get("eer", -epoch), 0
            torch.save({"model": model.state_dict(), "epoch": epoch + 1, "eer": rec.get("eer")},
                       os.path.join(run_dir, "best.pt"))
        else:
            bad += 1
        history.append(rec)
        path = os.path.join(run_dir, "checkpoints", f"epoch_{epoch + 1:03d}.pt")
        torch.save({"epoch": epoch, "model": model.state_dict(), "head": head.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "scaler": scaler.state_dict(), "history": history, "best": best,
                    "bad": bad, "torch_rng": torch.get_rng_state(),
                    "cuda_rng": torch.cuda.get_rng_state_all() if device == "cuda" else None}, path + ".tmp")
        os.replace(path + ".tmp", path)
        for old in sorted(glob.glob(os.path.join(run_dir, "checkpoints", "epoch_*.pt")))[:-3]:
            os.remove(old)
        log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
        if stop_after is not None and epoch + 1 >= stop_after:
            break
        if bad >= t["patience"]:
            log("Early stopping: identity EER stopped improving")
            break
    return model, history

def load_best(cfg, run_dir, device):
    model = build_encoder(cfg).to(device)
    st = torch.load(os.path.join(run_dir, "best.pt"), map_location=device, weights_only=False)
    model.load_state_dict(st["model"])
    return model.eval(), st

### Cell 2.4: Evaluation code: evaluate.py

In [ ]:
%%writefile /content/drive/MyDrive/ecg_leak/src/ecgleak/evaluate.py
import warnings
import numpy as np, torch, torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

@torch.no_grad()
def embed(model, signals, window, device, batch=256):
    """Template = mean of the embeddings of non-overlapping windows, L2-normalised."""
    model.eval()
    n_win = signals.shape[1] // window
    out = []
    for i in range(0, len(signals), batch):
        x = torch.from_numpy(signals[i:i + batch, :n_win * window].astype(np.float32)).to(device)
        e = model(x.reshape(-1, window)).reshape(-1, n_win, model.emb_dim).mean(1)
        out.append(F.normalize(e, dim=1).cpu().numpy())
    return np.concatenate(out)

def unit_rows(X):
    X = (X - X.mean(1, keepdims=True)) / (X.std(1, keepdims=True) + 1e-8)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)

def eer(genuine, impostor):
    y = np.r_[np.ones(len(genuine)), np.zeros(len(impostor))]
    fpr, tpr, _ = roc_curve(y, np.r_[genuine, impostor])
    i = np.nanargmin(np.abs((1 - tpr) - fpr))
    return float((fpr[i] + 1 - tpr[i]) / 2)

def verification(emb, patient_ids, dates):
    """Enroll each patient's earliest record; verify their later records against every enrolled patient."""
    order = np.lexsort((dates.astype("datetime64[ns]").astype(np.int64), patient_ids))
    seen, enroll, probe = set(), [], []
    for i in order:
        (probe if patient_ids[i] in seen else enroll).append(i)
        seen.add(patient_ids[i])
    E, P = emb[enroll], emb[probe]
    S = P @ E.T
    same = patient_ids[probe][:, None] == patient_ids[enroll][None, :]
    rank1 = float(np.mean(patient_ids[enroll][S.argmax(1)] == patient_ids[probe]))
    return {"eer": eer(S[same], S[~same]), "rank1": rank1, "n_enrolled": len(enroll), "n_probes": len(probe)}

def task_data(labels, names, cls):
    """Positives carry the class; negatives are NORM-only records."""
    names = list(names)
    pos = labels[:, names.index(cls)] == 1
    neg = (labels[:, names.index("NORM")] == 1) & (labels.sum(1) == 1)
    keep = pos | neg
    return keep, pos[keep].astype(int)

def demo_features(age, sex):
    a = np.where(np.isfinite(age), np.clip(age, 0, 90), np.nan)
    a = np.where(np.isnan(a), np.nanmedian(a), a)
    s = np.where(np.isfinite(sex), sex, 0.5)
    return np.c_[a, s]

def cv_auroc(X, y, folds, C_grid, pca=None, seed=0):
    """Out-of-fold AUROC over the given patient folds; C chosen by inner 3-fold CV on training folds only."""
    oof = np.full(len(y), np.nan)
    for f in np.unique(folds):
        tr, te = folds != f, folds == f
        steps = [StandardScaler()]
        if pca:
            steps.append(PCA(n_components=min(pca, X.shape[1], int(tr.sum()) - 1), random_state=seed))
        steps.append(LogisticRegressionCV(Cs=list(C_grid), cv=3, scoring="roc_auc", max_iter=3000))
        model = make_pipeline(*steps).fit(X[tr], y[tr])
        oof[te] = model.predict_proba(X[te])[:, 1]
    return float(roc_auc_score(y, oof)), oof

### Cell 2.5: Quick self-tests

In [ ]:
import ecgleak.models as M, ecgleak.train as TR, ecgleak.evaluate as EV
for mod in (M, TR, EV):
    importlib.reload(mod)
WINDOW = int(CFG["preprocess"]["window_s"] * CFG["data"]["fs"])

tests = {}
enc = M.build_encoder(CFG).to(DEVICE).eval()
with torch.no_grad():
    out = enc(torch.randn(3, WINDOW, device=DEVICE))
tests["Template is 256-d unit vector"] = out.shape == (3, CFG["model"]["emb_dim"]) and torch.allclose(out.norm(dim=1), torch.ones(3, device=DEVICE), atol=1e-4)
a = EV.embed(M.untrained_copies(CFG, [100], DEVICE)[0], D["id_eval"]["signal"][:8], WINDOW, DEVICE)
b = EV.embed(M.untrained_copies(CFG, [100], DEVICE)[0], D["id_eval"]["signal"][:8], WINDOW, DEVICE)
tests["Untrained copy is reproducible"] = np.allclose(a, b, atol=1e-5)
head = M.ArcFaceHead(CFG["model"]["emb_dim"], 5).to(DEVICE)
tests["ArcFace output finite"] = bool(torch.isfinite(head(out[:3], torch.tensor([0, 1, 2], device=DEVICE))).all())
y = np.r_[np.ones(500), np.zeros(500)]
tests["Shuffled-label probe ~ 0.5"] = 0.40 < EV.cv_auroc(np.random.default_rng(0).normal(size=(1000, 8)),
                                                       y, np.arange(1000) % 5, [1.0])[0] < 0.60
tests["Splits disjoint (no shared patient)"] = not (set(D["id_train"]["patient_id"]) & set(D["id_eval"]["patient_id"])
                                                    or set(D["id_train"]["patient_id"]) & set(D["probe_dev"]["patient_id"]))
for k, v in tests.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
TESTS_OK = all(tests.values())

### Cell 2.6: Identity baseline without deep learning

In [ ]:
mb = D["id_eval"]["median_beat"]
base_id = EV.verification(EV.unit_rows(mb), D["id_eval"]["patient_id"], D["id_eval"]["date"])
print(f"Identity baseline (median beat, nearest match) on ID-eval: EER {base_id['eer']:.3f} | "
      f"rank-1 {base_id['rank1']:.3f} | {base_id['n_enrolled']} enrolled, {base_id['n_probes']} probes")

### Cell 2.7: Diagnosis baselines: raw beat, age + sex, shuffled labels

In [ ]:
pdv = D["probe_dev"]
rows = []
for cls in CFG["leakage"]["tasks"]:
    keep, y = EV.task_data(pdv["labels"], SUPER, cls)
    folds = pdv["fold"][keep]
    if y.sum() < 10 or (1 - y).sum() < 10:
        print(f"{cls}: too few records, skipped"); continue
    raw_auc, _ = EV.cv_auroc(pdv["median_beat"][keep], y, folds, CFG["leakage"]["C_grid"], pca=64)
    demo_auc, _ = EV.cv_auroc(EV.demo_features(pdv["age"][keep], pdv["sex"][keep]), y, folds, CFG["leakage"]["C_grid"])
    shuf_auc, _ = EV.cv_auroc(pdv["median_beat"][keep], np.random.default_rng(1).permutation(y), folds,
                              CFG["leakage"]["C_grid"], pca=64)
    rows.append({"task": f"{cls} vs NORM", "n_pos": int(y.sum()), "n_neg": int((1 - y).sum()),
                 "raw_median_beat": raw_auc, "age_sex": demo_auc, "shuffled_control": shuf_auc})
baselines = pd.DataFrame(rows).set_index("task")
baselines.to_csv(f"{PATHS['results']}/baselines_probe_dev.csv")
print("Probe-dev AUROC (5-fold, patient-disjoint):\n" + baselines.round(3).to_string())

### Cell 2.8: Checkpoint resume test

In [ ]:
RT = "/content/resume_test"                                  # local scratch, not Drive
shutil.rmtree(RT, ignore_errors=True)
small = slice(0, min(512, len(D["id_train"]["signal"])))
sig, pid = D["id_train"]["signal"][small], D["id_train"]["patient_id"][small]
kw = dict(epochs=2, max_steps_per_epoch=3, log=lambda *_: None)
_, h_full = TR.train_identity(CFG, sig, pid, RT + "/full", DEVICE, **kw)
TR.train_identity(CFG, sig, pid, RT + "/split", DEVICE, stop_after=1, **kw)
_, h_res = TR.train_identity(CFG, sig, pid, RT + "/split", DEVICE, **kw)
RESUME_OK = len(h_res) == 2 and abs(h_res[-1]["loss"] - h_full[-1]["loss"]) <= 0.02 * abs(h_full[-1]["loss"]) + 1e-4
print(f"Uninterrupted epoch-2 loss {h_full[-1]['loss']:.4f} | stopped + resumed {h_res[-1]['loss']:.4f} -> "
      f"{'PASS' if RESUME_OK else 'FAIL'}")
shutil.rmtree(RT, ignore_errors=True)

### Cell 2.9: 5-epoch pilot identity model

In [ ]:
PILOT_DIR = f"{PATHS['runs']}/pilot_s0"
idv = D["id_eval"]
def id_eval_fn(model):
    r = EV.verification(EV.embed(model, idv["signal"], WINDOW, DEVICE), idv["patient_id"], idv["date"])
    return {"eer": r["eer"], "rank1": r["rank1"]}

ecfg.freeze(CFG, PILOT_DIR)
pilot_model, pilot_hist = TR.train_identity(CFG, D["id_train"]["signal"], D["id_train"]["patient_id"],
                                            PILOT_DIR, DEVICE, epochs=5, eval_fn=id_eval_fn)
pd.DataFrame(pilot_hist).to_csv(f"{PILOT_DIR}/history.csv", index=False)

### Cell 2.10: Pilot leak measurement on probe-dev

In [ ]:
pilot_best, st = TR.load_best(CFG, PILOT_DIR, DEVICE)
keep, y = EV.task_data(pdv["labels"], SUPER, CFG["leakage"]["primary"])
folds, C = pdv["fold"][keep], CFG["leakage"]["C_grid"]
sig_k = pdv["signal"][keep]

auc_id, _ = EV.cv_auroc(EV.embed(pilot_best, sig_k, WINDOW, DEVICE), y, folds, C)
auc_rand = [EV.cv_auroc(EV.embed(m, sig_k, WINDOW, DEVICE), y, folds, C)[0]
            for m in M.untrained_copies(CFG, CFG["leakage"]["untrained_seeds"], DEVICE)]
pilot = {"pilot_best_epoch": st["epoch"], "pilot_eer": st["eer"], "baseline_eer": base_id["eer"],
         "task": f"{CFG['leakage']['primary']} vs NORM", "auroc_pilot_template": auc_id,
         "auroc_untrained_mean": float(np.mean(auc_rand)), "auroc_untrained_each": auc_rand,
         "pilot_delta_rand": auc_id - float(np.mean(auc_rand))}
json.dump(pilot, open(f"{PATHS['results']}/phase2_pilot.json", "w"), indent=2)
print(f"Pilot identity EER {pilot['pilot_eer']:.3f} (baseline {base_id['eer']:.3f}) at epoch {st['epoch']}")
print(f"{pilot['task']} on probe-dev: pilot template AUROC {auc_id:.3f} | untrained copies {np.mean(auc_rand):.3f} "
      f"(each: {', '.join(f'{a:.3f}' for a in auc_rand)})")
print(f"Pilot Delta_rand = {pilot['pilot_delta_rand']:+.3f}  (pipeline check only: 5 epochs, not a result)")

### Cell 2.11: Phase 2 gate

In [ ]:
ctrl = baselines["shuffled_control"]
gate = {"Model/eval unit tests": TESTS_OK,
        "Identity baseline EER computed": np.isfinite(base_id["eer"]),
        "Raw-beat and age+sex baselines computed": len(baselines) > 0 and bool(baselines[["raw_median_beat", "age_sex"]].notna().all().all()),
        "Shuffled-label control near 0.5": bool(((ctrl > 0.40) & (ctrl < 0.60)).all()),
        "Checkpoint resume works": RESUME_OK,
        "Pilot trained and saved to Drive": os.path.isfile(f"{PILOT_DIR}/best.pt"),
        "Pilot Delta_rand computed end to end": np.isfinite(pilot["pilot_delta_rand"])}
GO = all(gate.values())
print("PHASE 2 GATE\n" + "-" * 50)
for k, v in gate.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("-" * 50)
print("GO: pipeline works end to end. Next: Phase 3 (full identity training + lockbox)." if GO
      else "NO-GO: fix the FAIL lines above.")
json.dump({"phase": 2, "go": GO, "checks": gate, "identity_baseline": base_id, "pilot": pilot,
           "baselines_probe_dev": baselines.round(4).reset_index().to_dict("records"),
           "environment": ecfg.environment()},
          open(f"{PATHS['gates']}/gate_2.json", "w"), indent=2, default=str)